<div align="center">

# 🚕 Yandex Go Analytics

## 8 · Security, Governance

*Least-privilege access, a machine-readable audit report, and the whole project as code.*

![security](https://img.shields.io/badge/security-Unity%20Catalog-FF3621)
![access](https://img.shields.io/badge/access-GRANT%20%2F%20REVOKE-7B1FA2)
![report](https://img.shields.io/badge/report-JSON-2E7D32)
![deploy](https://img.shields.io/badge/deploy-Asset%20Bundles-4B8BBE)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [7 · Refactor via Delta Live Tables](./7.%20Refactor%20via%20Delta%20Live%20Tables.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | — |

**Progress** &nbsp; ▰▰▰▰▰▰▰▰  **8 / 8**

---

## 🎯 Task requirements

1. [x] Create the workspace user **`innowise@innowise.com`** and add them to a new group **`innowise`**
2. [x] Grant the group **read** privileges (`SELECT` / `USAGE`) on the **Gold** schema, then **`REVOKE`** a specific privilege
3. [x] Programmatically generate a **JSON report** with row counts and execution results of the workflow
4. [x] *(Optional)* Package and deploy the whole pipeline with **Databricks Asset Bundles**

## 🧩 How it is solved

| # | Requirement | Mechanism |
|:--:|:--|:--|
| 1 | User and group | Databricks SDK (`WorkspaceClient`) — idempotent *create if missing* |
| 2 | Access control | Unity Catalog `GRANT` / `REVOKE` / `SHOW GRANTS` |
| 3 | JSON report | `WorkflowReportService` → Pydantic-validated `WorkflowAuditReport`, saved into the Volume |
| 4 | Packaging | `databricks.yml` + `resources/` deployed with `databricks bundle deploy` |

### 🔑 The privilege model

```text
 Catalog   <catalog>                    ◀── USE CATALOG   (can "see" the catalog)
   └─ Schema  <catalog>.gold            ◀── USE SCHEMA    (can "see" the schema)
        └─ Tables  taxi_metrics, …      ◀── SELECT        (can read the data; inherited from the schema)
```

| Privilege | Meaning |
|:--|:--|
| `USE CATALOG` / `USE SCHEMA` | Navigate to the object (the "usage" part of *read access*) |
| `SELECT` | Query the tables (the "read" part) |

## 1️⃣ Connect to Spark

In [1]:
from pyspark.sql import functions as F

from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-security-governance")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 15:28:50.573 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-security-governance | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_staging | env = staging


## 2️⃣ Create the user and the group

**Target:** user `innowise@innowise.com` ∈ group `innowise`.

<details>
<summary><b>🖱 Prefer the UI?</b></summary>

<br>

1. **Settings → Identity and access → Users → Add user** → `innowise@innowise.com`
2. **Settings → Identity and access → Groups → Add group** → `innowise`
3. Open the group → **Add members** → select `innowise@innowise.com`

</details>

> ⚠️ **Unity Catalog needs a group it can resolve.** If the `GRANT` below fails with *"principal does not exist"*,
> the group is workspace-local. Create it as an **account-level group** (Account console → User management →
> Groups), add the user there, and re-run.

## 3️⃣ Grant read access on the Gold schema

Three statements give the group *read-only* access:

```sql
GRANT USE CATALOG ON CATALOG <catalog>      TO `innowise`;
GRANT USE SCHEMA  ON SCHEMA  <catalog>.gold TO `innowise`;
GRANT SELECT      ON SCHEMA  <catalog>.gold TO `innowise`;
```

In [2]:
GOLD_SCHEMA = f"{CATALOG}.gold"

GROUP_NAME = "innowise"

grant_statements = [
    f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{GROUP_NAME}`",
    f"GRANT USE SCHEMA ON SCHEMA {GOLD_SCHEMA} TO `{GROUP_NAME}`",
    f"GRANT SELECT ON SCHEMA {GOLD_SCHEMA} TO `{GROUP_NAME}`",
]
for statement in grant_statements:
    print("▶", statement)
    spark.sql(statement)


def group_grants():
    """Grants of the `innowise` group on the Gold schema."""
    return spark.sql(f"SHOW GRANTS ON SCHEMA {GOLD_SCHEMA}").filter(F.col("Principal") == GROUP_NAME)


print(f"\nGrants for `{GROUP_NAME}` on {GOLD_SCHEMA}:")
display(group_grants().toPandas())

granted = {row["ActionType"] for row in group_grants().collect()}
assert {"USE SCHEMA", "SELECT"} <= granted, f"Unexpected privileges: {granted}"
print("✅ The group can now browse and read the Gold schema")

▶ GRANT USE CATALOG ON CATALOG yandex_go_staging TO `innowise`
▶ GRANT USE SCHEMA ON SCHEMA yandex_go_staging.gold TO `innowise`
▶ GRANT SELECT ON SCHEMA yandex_go_staging.gold TO `innowise`

Grants for `innowise` on yandex_go_staging.gold:


,Principal,ActionType,ObjectType,ObjectKey
0,innowise,SELECT,SCHEMA,yandex_go_staging.gold
1,innowise,USE SCHEMA,SCHEMA,yandex_go_staging.gold


✅ The group can now browse and read the Gold schema


## 4️⃣ Revoke a specific privilege

Take back **only `SELECT`**: the group can still *see* the schema (`USE SCHEMA`) but can no longer read the tables.

```sql
REVOKE SELECT ON SCHEMA <catalog>.gold FROM `innowise`;
```

In [3]:
revoke_statement = f"REVOKE SELECT ON SCHEMA {GOLD_SCHEMA} FROM `{GROUP_NAME}`"
print("▶", revoke_statement)
spark.sql(revoke_statement)

print(f"\nGrants for `{GROUP_NAME}` after REVOKE:")
display(group_grants().toPandas())

remaining = {row["ActionType"] for row in group_grants().collect()}
assert "SELECT" not in remaining, "SELECT must be revoked"
assert "USE SCHEMA" in remaining, "USE SCHEMA must stay untouched"
print(f"✅ SELECT revoked; remaining privileges: {sorted(remaining)}")

▶ REVOKE SELECT ON SCHEMA yandex_go_staging.gold FROM `innowise`

Grants for `innowise` after REVOKE:


,Principal,ActionType,ObjectType,ObjectKey
0,innowise,USE SCHEMA,SCHEMA,yandex_go_staging.gold


✅ SELECT revoked; remaining privileges: ['USE SCHEMA']


> 💡 To restore read access, re-run the `GRANT` cell above. Note that `resources/schemas.yml` also declares
> `USE_SCHEMA` + `SELECT` for `innowise`, so the next `databricks bundle deploy` re-applies both.

## 5️⃣ JSON audit report

`WorkflowReportService` inspects all **11 tables** of the medallion and records, for each one:

| Field | Meaning |
|:--|:--|
| `exists`, `status` | `HEALTHY` · `NOT_FOUND` · `ERROR` |
| `row_count` | Number of rows |
| `latest_version`, `latest_operation`, `last_modified` | Latest entry of the Delta history |

The result is validated with **Pydantic** and saved as JSON in the Volume:
`/Volumes/<catalog>/raw_files/landing/reports/workflow_summary.json`.

In [4]:
from src.core.infra import WorkflowReportService

reporter = WorkflowReportService()
report = reporter.generate_report(spark)

line = "=" * 108
print(line)
print(f"LAKEHOUSE AUDIT REPORT | Environment: {report.environment.upper()} | Status: {report.summary.pipeline_status}")
print(
    f"Catalog: {report.catalog} | Tables: {report.summary.total_monitored_tables} "
    f"| Total records: {report.summary.total_rows_across_lakehouse:,}"
)
print(line)
print(f"{'LAYER':<8} | {'TABLE':<16} | {'ROWS':>8} | {'VERSION':<8} | {'OPERATION':<18} | {'STATUS':<10} | LAST MODIFIED")
print("-" * 108)

for layer, tables in report.layers.items():
    for table_name, meta in tables.items():
        version = f"v{meta.latest_version}" if meta.latest_version is not None else "-"
        print(
            f"{layer.upper():<8} | {table_name:<16} | {meta.row_count:>8,} | {version:<8} | "
            f"{(meta.latest_operation or '-'):<18} | {meta.status.value:<10} | {meta.last_modified or '-'}"
        )
print(line)

unhealthy = [
    f"{layer}.{name}"
    for layer, tables in report.layers.items()
    for name, meta in tables.items()
    if meta.status.value != "HEALTHY"
]
assert not unhealthy, f"Unhealthy tables: {unhealthy}"
print("✅ All monitored tables are HEALTHY")

2026-10-09 15:28:56.116 | INFO     | src.core.infra.audit.reporting:generate_report:97 - Generating Pydantic-validated Audit Report for [yandex_go_staging]...
2026-10-09 15:29:24.198 | INFO     | src.core.infra.audit.reporting:generate_report:134 - Audit report successfully saved to: /Volumes/yandex_go_staging/raw_files/landing/reports/workflow_summary.json


LAKEHOUSE AUDIT REPORT | Environment: STAGING | Status: SUCCESS
Catalog: yandex_go_staging | Tables: 10 | Total records: 7,999
LAYER    | TABLE            |     ROWS | VERSION  | OPERATION          | STATUS     | LAST MODIFIED
------------------------------------------------------------------------------------------------------------
BRONZE   | taxi             |    2,000 | v1       | STREAMING UPDATE   | HEALTHY    | 2026-10-09 15:21:29
BRONZE   | drivers          |       50 | v1       | STREAMING UPDATE   | HEALTHY    | 2026-10-09 15:21:56
BRONZE   | users            |      500 | v1       | STREAMING UPDATE   | HEALTHY    | 2026-10-09 15:21:17
SILVER   | taxi             |    1,913 | v2       | OPTIMIZE           | HEALTHY    | 2026-10-09 15:23:10
SILVER   | drivers          |       37 | v2       | OPTIMIZE           | HEALTHY    | 2026-10-09 15:23:28
SILVER   | users            |      458 | v2       | OPTIMIZE           | HEALTHY    | 2026-10-09 15:23:52
GOLD     | taxi_metrics     

### 🧾 The JSON document

In [5]:
print(report.model_dump_json(indent=2))

{
  "pipeline_name": "yandex-go-analytics",
  "environment": "staging",
  "catalog": "yandex_go_staging",
  "generated_at": "2026-10-09T12:29:21.541991Z",
  "summary": {
    "total_monitored_tables": 10,
    "total_rows_across_lakehouse": 7999,
    "pipeline_status": "SUCCESS"
  },
  "layers": {
    "bronze": {
      "taxi": {
        "exists": true,
        "row_count": 2000,
        "latest_version": 1,
        "latest_operation": "STREAMING UPDATE",
        "last_modified": "2026-10-09 15:21:29",
        "status": "HEALTHY",
        "error": null
      },
      "drivers": {
        "exists": true,
        "row_count": 50,
        "latest_version": 1,
        "latest_operation": "STREAMING UPDATE",
        "last_modified": "2026-10-09 15:21:56",
        "status": "HEALTHY",
        "error": null
      },
      "users": {
        "exists": true,
        "row_count": 500,
        "latest_version": 1,
        "latest_operation": "STREAMING UPDATE",
        "last_modified": "2026-10-09 1

## 6️⃣ Optional · Package & deploy with Databricks Asset Bundles

The **whole** project — infrastructure, pipeline, job and permissions — is described as code and deployed with a
single command.

```text
 databricks.yml                         ← variables + targets (dev / staging / prod)
 resources/
 ├── schemas.yml                        ← raw_files · bronze · silver · gold  (+ grants for `innowise`)
 ├── volumes.yml                        ← landing volume
 ├── pipelines/yandex_go_trips_pipeline.yml   ← DLT pipeline (serverless)
 ├── jobs/scheduled_tripes_pipeline_job.yml   ← daily trigger of the pipeline
 └── dashboards/                        ← Lakeview dashboards (prepared)
```

### 🌍 Targets

| Target | Catalog | Mode | Deploy |
|:--|:--|:--:|:--|
| `dev` *(default)* | `yandex_go_dev` | – | `make deploy-dev` |
| `staging` | `yandex_go_staging` | production | `make deploy TARGET=staging` |
| `prod` | `yandex_go_prod` | production | `make deploy-prod` |

### ⌨️ Commands

```bash
# 1 · validate the configuration
make validate                       # databricks bundle validate -t dev

# 2 · deploy schemas, volume, pipeline and job
make deploy-dev                     # databricks bundle deploy -t dev

# 3 · run the DLT pipeline
make run-trips-dev                  # databricks bundle run yandex_go_trips_pipeline -t dev

# 4 · inspect what is deployed
databricks bundle summary -t dev
```

### 🔁 CI/CD

| Workflow | Trigger | Does |
|:--|:--|:--|
| **Code Quality & Tests** | Pull request | Ruff, MyPy, unit + integration tests, `bundle validate` |
| **Deploy to Target** | Push to a branch, manual | `databricks bundle deploy` to the matching target |
| **Security Scans** | Pull request, push, weekly | Gitleaks, pip-audit, Bandit |
| **Documentation** | Pull request, push to `production` | Strict MkDocs build, publish to GitHub Pages |

## 🏁 Final scoreboard

| # | Task section | Notebook | Status |
|:--:|:--|:--|:--:|
| 1 | Setup — schemas, Volume, source files | [Setup](./1.%20Setup.ipynb) | ✅ |
| 2 | Batch ingestion | [Batch Ingestion](./2.%20Batch%20Ingestion.ipynb) | ✅ |
| 3 | Streaming, schema evolution, time travel | [Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb) | ✅ |
| 4 | Data cleansing | [Data Cleaning](./4.%20Data%20Cleaning.ipynb) | ✅ |
| 5 | Aggregations & optimisations | [Aggregations & Optimizations](./5.%20Aggregations%20%26%20Optimizations.ipynb) | ✅ |
| 6 | Data enrichment | [Data Enrichment](./6.%20Data%20Enrichment.ipynb) | ✅ |
| 7 | Delta Live Tables | [Refactor via Delta Live Tables](./7.%20Refactor%20via%20Delta%20Live%20Tables.ipynb) | ✅ |
| 8 | Security, governance, report, DABs | this notebook | ✅ |

---

## ✅ Checkpoint

- [x] User `innowise@innowise.com` and group `innowise` created
- [x] `USE CATALOG`, `USE SCHEMA`, `SELECT` granted on the Gold schema; `SELECT` revoked
- [x] JSON audit report generated, validated and stored in the Volume
- [x] Project packaged and deployed with Databricks Asset Bundles

> 🎉 the lakehouse is built, governed and reproducible. Head back to the overview any time.

<div align="center"><sub>Yandex Go Analytics · notebook 8 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>